# Notebook 04 — Seller Master Creation

**Purpose:**
Build `company_info`, the master reference table of every unique company in the dataset.
Merge GEM tech eval data (MSE/MII status, tech eval status) into financial_eval.
Assign a stable `company_id` to each company and link FKs into financial_eval.

**Inputs** (`data/processed/`):
- `final_bid_data.xlsx`
- `final_FE.xlsx` (with `drilling_meterage` and `per_meter_rate`)

**Also reads** (`data/raw/`, if available):
- `gem_specific_tech_eval_output.xlsx` — GEM tech eval fields

**Outputs** (`data/processed/`):
- `company_info.xlsx`
- `final_FE.xlsx` — updated with `fe_id`, `seller_company_id`, tech eval columns

**Schema — company_info:**

| Column | Type | Description |
|--------|------|-------------|
| `company_id` | str | Stable sequential ID (`CO_00001`) |
| `company_name` | str | Cleaned title-case display name |
| `company_valuation_cr` | float | Valuation in Crore INR (manual enrichment) |
| `swot` | str | SWOT notes (manual enrichment) |
| `yoy_growth_percent` | float | YoY revenue growth % (manual enrichment) |

---

## 1. Imports & Setup

In [1]:
import sys
import uuid
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from config import RAW_FILES, PROCESSED_FILES, DATA_RAW
from src.cleaning import clean_company_name

print('Setup complete.')

Setup complete.


## 2. Load Data

In [2]:
bid_data = pd.read_excel(PROCESSED_FILES['featured_bids'])
fe_data  = pd.read_excel(PROCESSED_FILES['merged_fe'])

bid_data.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
fe_data.drop( columns=['Unnamed: 0'], inplace=True, errors='ignore')

if bid_data.index.name == 'bid_number':
    bid_data = bid_data.reset_index()

print(f"bid_data : {bid_data.shape}")
print(f"fe_data  : {fe_data.shape}")

bid_data : (5689, 16)
fe_data  : (22519, 7)


In [3]:
# Guard: company_id linkage below keys on bid_number / seller pairs.
# A non-unique bid_number would let one company's win get attributed to another.
assert bid_data['bid_number'].is_unique, (
    "bid_number is not unique in final_bid_data.xlsx -- check Notebook 02/03 output."
)
print("Guard passed: bid_number is unique.")

Guard passed: bid_number is unique.


## 3. Clean Financial Evaluation

In [4]:
n_start = len(fe_data)

fe_data = fe_data.dropna(subset=['bid_number', 'seller', 'price']).copy()
fe_data['seller']        = fe_data['seller'].astype(str).str.strip().apply(clean_company_name)
fe_data['rank_position'] = fe_data['rank_position'].astype(str).str.strip()
fe_data = fe_data.reset_index(drop=True)

print(f"FE rows before cleaning : {n_start}")
print(f"FE rows after  cleaning : {len(fe_data)}")

FE rows before cleaning : 22519
FE rows after  cleaning : 22516


## 4. Merge GEM Tech Eval Data (if available)

Merges `Participated On`, `MSE/MII Status`, and `Tech Eval Status` from the
GEM-specific tech eval file into financial_eval on `(bid_number, seller)`.

**Fix note:** an earlier version of this cell crashed with
`ValueError: cannot reindex on an axis with duplicate labels` when the raw
tech eval file contained duplicate `(bid_number, seller)` rows. It also let
unclean names from `tech_eval` leak into `seller` without passing through
`clean_company_name`, which produced inconsistent variants like
`"mapp private limited"` / `"MAPP pvt Ltd."` / `"company Pvt Ltd"` for what
should be the same company. Both issues are fixed below: duplicates are
dropped explicitly before merging, and every recovered name is re-cleaned.

In [5]:
tech_eval_path = DATA_RAW / 'gem_specific_tech_eval_output.xlsx'

if tech_eval_path.exists():
    tech_eval = pd.read_excel(tech_eval_path)
    tech_eval['Seller Name'] = tech_eval['Seller Name'].apply(clean_company_name)
    tech_eval = tech_eval.dropna(subset=['Seller Name'])

    # Build normalised join keys (case-insensitive, whitespace-safe)
    fe_data['_join_key'] = (
        fe_data['bid_number'].astype(str) + '||' +
        fe_data['seller'].astype(str).str.strip().str.upper()
    )
    tech_eval['_join_key'] = (
        tech_eval['Bid No'].astype(str) + '||' +
        tech_eval['Seller Name'].astype(str).str.strip().str.upper()
    )

    # Tech eval rows must be unique per join key before merging -- duplicates
    # here previously caused a "cannot reindex on an axis with duplicate
    # labels" crash. Drop dupes explicitly and report how many were found.
    n_before = len(tech_eval)
    tech_eval = tech_eval.drop_duplicates(subset='_join_key', keep='first')
    n_dropped = n_before - len(tech_eval)
    if n_dropped > 0:
        print(f"Dropped {n_dropped} duplicate (bid_number, seller) rows from tech_eval source file.")

    tech_cols = ['_join_key', 'Bid No', 'Seller Name', 'Participated On', 'MSE/MII Status', 'Tech Eval Status']
    fe_data = fe_data.merge(
        tech_eval[tech_cols],
        on='_join_key', how='outer'
    )

    # Recover bid_number / seller for rows that only existed in tech_eval
    # (outer-join-only rows), using columns brought in by the merge itself --
    # no separate set_index/reindex needed, which is what crashed before.
    fe_data['bid_number'] = fe_data['bid_number'].fillna(fe_data['Bid No'])
    fe_data['seller']     = fe_data['seller'].fillna(fe_data['Seller Name'])

    # Run every recovered name through the same cleaner as the rest of the
    # pipeline, so names arriving via this merge cannot bypass normalisation
    # and reintroduce inconsistent Pvt/Private/Ltd/Limited variants.
    fe_data['seller'] = fe_data['seller'].apply(clean_company_name)

    fe_data.drop(columns=['_join_key', 'Bid No', 'Seller Name'], inplace=True)
    print(f"Tech eval merged. FE shape: {fe_data.shape}")
else:
    fe_data['Participated On']  = np.nan
    fe_data['MSE/MII Status']   = np.nan
    fe_data['Tech Eval Status'] = np.nan
    print("gem_specific_tech_eval_output.xlsx not found -- tech eval columns set to NaN.")

Dropped 5 duplicate (bid_number, seller) rows from tech_eval source file.
Tech eval merged. FE shape: (29203, 10)


## 5. Verify Name Cleaning Worked

Quick check: no two distinct raw strings should still collapse to visibly
different cleaned names for what is clearly the same company. This is a
spot-check, not exhaustive -- it flags near-duplicate names for manual review.

In [6]:
# Spot-check: look for names that differ only by case/whitespace after cleaning,
# which would indicate clean_company_name was skipped somewhere upstream.
seller_counts = fe_data['seller'].value_counts()
lower_groups  = fe_data['seller'].dropna().str.lower().value_counts()

suspicious = lower_groups[lower_groups.index.duplicated(keep=False)]
distinct_cased_variants = (
    fe_data[['seller']]
    .dropna()
    .assign(lower=lambda d: d['seller'].str.lower())
    .groupby('lower')['seller']
    .nunique()
)
flagged = distinct_cased_variants[distinct_cased_variants > 1]

if len(flagged) > 0:
    print(f"Warning: {len(flagged)} company names still have multiple casing variants after cleaning:")
    for lower_name in flagged.index[:10]:
        variants = fe_data.loc[fe_data['seller'].str.lower() == lower_name, 'seller'].unique()
        print(f"  {list(variants)}")
else:
    print("No casing inconsistencies found -- clean_company_name applied uniformly.")

No casing inconsistencies found -- clean_company_name applied uniformly.


## 6. Build Seller Universe

Union of all `seller` names in fe_data and all `winner` names in bid_data,
ensuring every FK reference can be resolved.

In [7]:
fe_names  = set(fe_data['seller'].dropna().unique())
win_names = set(bid_data['winner'].dropna().unique())
all_names = fe_names | win_names

print(f"Unique sellers in financial_eval : {len(fe_names)}")
print(f"Unique winners  in bid_data      : {len(win_names)}")
print(f"Winners not in financial_eval    : {len(win_names - fe_names)}")
print(f"Total unique companies           : {len(all_names)}")

Unique sellers in financial_eval : 6828
Unique winners  in bid_data      : 2262
Winners not in financial_eval    : 19
Total unique companies           : 6847


## 7. Construct company_info Table

In [8]:
company_info = (
    pd.DataFrame({'company_name': sorted(all_names)})
    .drop_duplicates(subset=['company_name'])
    .reset_index(drop=True)
)

company_info['company_id']           = [f'CO_{i:05d}' for i in range(1, len(company_info) + 1)]
company_info['company_valuation_cr'] = np.nan
company_info['swot']                 = np.nan
company_info['yoy_growth_percent']   = np.nan

company_info = company_info[[
    'company_id', 'company_name',
    'company_valuation_cr', 'swot', 'yoy_growth_percent'
]].dropna(subset=['company_name']).reset_index(drop=True)

print(f"company_info shape: {company_info.shape}")

company_info shape: (6847, 5)


## 8. Link company_id into Financial Eval

In [9]:
name_to_id: dict = (
    company_info
    .dropna(subset=['company_name'])
    .assign(key=lambda x: x['company_name'].str.strip().str.lower())
    .drop_duplicates(subset='key', keep='first')
    .set_index('key')['company_id']
    .to_dict()
)

fe_data['company_id'] = (
    fe_data['seller'].str.strip().str.lower().map(name_to_id)
)

unmatched = fe_data[fe_data['company_id'].isna()]['seller'].drop_duplicates().sort_values()
if len(unmatched) > 0:
    print(f"Warning: {len(unmatched)} unmatched sellers in financial_eval:")
    print(unmatched.to_string())
else:
    print("All sellers resolved to company_id successfully.")

All sellers resolved to company_id successfully.


## 9. Assign fe_id & Finalise Column Order

In [10]:
fe_data['fe_id'] = [
    'FE_' + str(uuid.uuid4())[:8].upper() for _ in range(len(fe_data))
]

financial_eval = pd.DataFrame()
financial_eval[[
    'financial_eval_id',
    'bid_number',
    'seller_company_id',
    'rank_position',
    'seller',
    'price',
    'status',
    'drilling_meterage',
    'per_meter_rate',
    'Participated On',
    'MSE/MII Status',
    'Tech Eval Status',
]] = fe_data[[
    'fe_id', 'bid_number', 'company_id', 'rank_position', 'seller', 'price', 'status',
    'drilling_meterage', 'per_meter_rate',
    'Participated On', 'MSE/MII Status', 'Tech Eval Status',
]]

print(f"final_financial_eval shape: {financial_eval.shape}")

final_financial_eval shape: (29203, 12)


## 10. Export

In [11]:
company_info.to_excel(PROCESSED_FILES['company_info'], index=False)

financial_eval.set_index('financial_eval_id', drop=True, inplace=True)
financial_eval.to_excel(PROCESSED_FILES['final_fe'])

print(f"Exported company_info  ->  {PROCESSED_FILES['company_info'].name}  ({len(company_info)} rows)")
print(f"Exported final_FE      ->  {PROCESSED_FILES['final_fe'].name}  ({len(financial_eval)} rows)")
print("\nNotebook 04 complete. Proceed to 05_competitor_analysis.ipynb.")

Exported company_info  ->  company_info.xlsx  (6847 rows)
Exported final_FE      ->  final_FE.xlsx  (29203 rows)

Notebook 04 complete. Proceed to 05_competitor_analysis.ipynb.
